# Error Budget & Uncertainty Ellipse Combination**Source**: `grdl/example/shapes/error_budget_overlay.py`## Learning Objectives- Define uncertainty ellipses for independent error sources- Combine ellipses via covariance convolution- Overlay 1σ/2σ/3σ confidence regions on imagery- Understand error propagation in geolocation workflows## Theory: Error Ellipse Convolution**Problem**: A detection is localized with multiple independent error sources (pixel accuracy, sensor pointing, georegistration). What is the **total uncertainty**?**Solution**: Each error source is modeled as a 2D Gaussian uncertainty ellipse (semi-major axis, semi-minor axis, rotation). Independent sources are combined via **covariance matrix addition**:$$\Sigma_{\text{total}} = \Sigma_1 + \Sigma_2 + \Sigma_3 + \dots$$**GRDL workflow**: `convolve_ellipses([ellipse1, ellipse2, ...])` → combined ellipse**Common error sources**:- **Pixel localization**: Target centroid estimation error (isotropic, small)- **Sensor pointing**: Attitude/ephemeris error (anisotropic, aligned with look direction)- **Georegistration**: DEM/geoid error (aligned east-north)**Physical intuition**:- 1σ ellipse: 39% probability target is inside- 2σ ellipse: 86% probability- 3σ ellipse: 99% probability## Data SetupThis example uses a **blank synthetic scene** to demonstrate the math — no imagery needed.

In [ ]:
"""GRDL Notebook Preamble — Environment validation and autoreload."""
try:
    get_ipython().run_line_magic('load_ext', 'autoreload')
    get_ipython().run_line_magic('autoreload', '2')
except (NameError, AttributeError):
    pass  # Not running in IPython

# Validate GRDL version
import grdl
from packaging.version import Version

REQUIRED_VERSION = '0.6.1'
current_version = Version(grdl.__version__)
required_version = Version(REQUIRED_VERSION)

if current_version < required_version:
    raise RuntimeError(
        f"GRDL {REQUIRED_VERSION}+ required. Found {grdl.__version__}. "
        f"Update with: pip install --upgrade grdl"
    )

print(f"✓ GRDL {grdl.__version__} (>= {REQUIRED_VERSION})")

In [ ]:
import numpy as npimport matplotlib.pyplot as pltfrom rasterio.transform import Affinefrom grdl.geolocation.eo.affine import AffineGeolocationfrom grdl.shapes import Ellipse, convolve_ellipses, overlay_shape

In [ ]:
# Create blank scene and geolocationshape = (512, 512)image = np.zeros(shape, dtype=np.float64)transform = Affine(1e-3, 0.0, -118.2, 0.0, -1e-3, 34.1)geo = AffineGeolocation(transform, shape, 'EPSG:4326')print(f"Scene shape: {shape}")print(f"GSD: {1e-3 * 111_000:.1f} m/pixel (at equator)")

In [ ]:
# Detection location (arbitrary point)det_lat, det_lon = 34.05, -118.15print(f"Detection location: ({det_lat:.6f}°, {det_lon:.6f}°)")# Verify pixel coordinatespixel_coords = geo.latlon_to_image(np.array([[det_lat, det_lon]]))print(f"  → Pixel: ({pixel_coords[0, 0]:.1f}, {pixel_coords[0, 1]:.1f})")

In [ ]:
# Define three independent error sourcessources = [    Ellipse(det_lat, det_lon, semi_major_m=8.0, semi_minor_m=8.0, rotation_deg=0.0),    Ellipse(det_lat, det_lon, semi_major_m=40.0, semi_minor_m=20.0, rotation_deg=30.0),    Ellipse(det_lat, det_lon, semi_major_m=25.0, semi_minor_m=15.0, rotation_deg=0.0),]labels = ['Pixel localisation', 'Sensor pointing', 'Georegistration']colors = ['yellow', 'magenta', 'cyan']print("Error sources:")for label, src in zip(labels, sources):    print(f"  {label}: a={src.semi_major_m:.1f} m, b={src.semi_minor_m:.1f} m, rot={src.rotation_deg:.1f}°")

In [ ]:
# Convolve ellipses → total 1-sigma uncertaintycombined_1s = convolve_ellipses(sources)print(f"\nCombined 1-sigma ellipse:")print(f"  Semi-major: {combined_1s.semi_major_m:.2f} m")print(f"  Semi-minor: {combined_1s.semi_minor_m:.2f} m")print(f"  Rotation: {combined_1s.rotation_deg:.1f}°")print(f"  Area: {np.pi * combined_1s.semi_major_m * combined_1s.semi_minor_m:.0f} m²")# Derive 2-sigma and 3-sigma ellipses (scale axes by 2x and 3x)combined_2s = Ellipse(    det_lat, det_lon,    semi_major_m=combined_1s.semi_major_m * 2.0,    semi_minor_m=combined_1s.semi_minor_m * 2.0,    rotation_deg=combined_1s.rotation_deg,)combined_3s = Ellipse(    det_lat, det_lon,    semi_major_m=combined_1s.semi_major_m * 3.0,    semi_minor_m=combined_1s.semi_minor_m * 3.0,    rotation_deg=combined_1s.rotation_deg,)print(f"  2-sigma area: {np.pi * combined_2s.semi_major_m * combined_2s.semi_minor_m:.0f} m²")print(f"  3-sigma area: {np.pi * combined_3s.semi_major_m * combined_3s.semi_minor_m:.0f} m²")

In [ ]:
# Visualize: overlay individual sources + combined 1σ/2σ/3σfig, ax = plt.subplots(figsize=(10, 10))ax.imshow(image, cmap='gray', vmin=0, vmax=1)# Overlay individual error sources (thin lines)for src, label, color in zip(sources, labels, colors):    overlay_shape(ax, src, geo, color=color, linewidth=1.5, label=label, linestyle='--')# Overlay combined ellipses (thick lines)for sigma_shape, color, label in [    (combined_1s, 'lime', '1-sigma total (39%)'),    (combined_2s, 'orange', '2-sigma total (86%)'),    (combined_3s, 'red', '3-sigma total (99%)'),]:    overlay_shape(ax, sigma_shape, geo, color=color, linewidth=2.5, label=label)ax.set_title('Error Budget Overlay: Independent Sources + Convolved Total', fontsize=14)ax.legend(loc='upper right', fontsize=10)plt.tight_layout()plt.show()

## Physical Interpretation**Observation**: The combined 1-sigma ellipse is **larger** than any individual source but **smaller** than their sum.**Why**: Error convolution is **quadratic** (covariance addition), not linear:$$\sigma_{\text{total}} = \sqrt{\sigma_1^2 + \sigma_2^2 + \sigma_3^2}$$**Example**: 8 m + 40 m + 25 m ≠ 73 m (linear)  Instead: √(8² + 40² + 25²) ≈ 48 m (quadratic)**Key insight**: Independent errors combine in quadrature — the total error is dominated by the largest source, not the sum.**Rotation dominance**: The combined ellipse's orientation is determined by the largest anisotropic source (sensor pointing at 30°).## When to Use Error Ellipses**Typical workflow**:1. Define per-source ellipses (from sensor calibration, DEM accuracy, etc.)2. Convolve via `convolve_ellipses()` → total 1σ uncertainty3. Scale to 2σ/3σ for confidence regions4. Overlay on imagery to assess detection quality**Applications**:- **Search and rescue**: Define search area around last known position- **Target mensuration**: Report position ± uncertainty ellipse- **Multi-source fusion**: Weight detections by inverse covariance## Summary**GRDL patterns demonstrated**:- ✅ `grdl.shapes.Ellipse` — 2D Gaussian uncertainty ellipse- ✅ `grdl.shapes.convolve_ellipses()` — covariance convolution- ✅ `grdl.shapes.overlay_shape()` — matplotlib ellipse overlay**Math**: Independent ellipses combine via $\Sigma_{\text{total}} = \sum_i \Sigma_i$ (covariance addition)**Physical interpretation**: Combined error is quadratic, not linear — dominated by largest source.